# Kirsch Visualizer — Near-Wellbore Stresses

Interactive-style demo of the **Kirsch solution** for stresses on the borehole wall.

Uses `NearWellboreStressesCalculation.calculate_kirsch_borehole_wall_stresses`
and `calculate_principal_stresses_analytical` from GeomechPy.

Polar plots show radial, tangential and axial stress components around the wellbore circumference.


## Setup & imports


In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == "Project":
    REPO_ROOT = REPO_ROOT.parent.parent
elif REPO_ROOT.name == "example":
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from geomechpy.near_wellbore_stresses import NearWellboreStressesCalculation

print("Imports OK — repo root:", REPO_ROOT)


## Input parameters (single depth)

Edit these values to explore different stress regimes and well trajectories.


In [ ]:
# Far-field stresses (psi)
SHMIN = 6500.0
SHMAX = 8500.0
SVERT = 10000.0
PORE_PRESSURE = 4500.0
MUD_PRESSURE = 5000.0

# Orientation
SHMAX_AZIMUTH = 30.0       # deg from geographic North
BOREHOLE_DEVIATION = 0.0   # 0 = vertical well
BOREHOLE_AZIMUTH = 0.0     # deg

# Rock property
POISSON_RATIO_STATIC = 0.25

# Circumferential sampling (deg relative to Top-of-Hole)
theta = np.linspace(0, 360, 361)

print(f"Stress regime check: Sv={SVERT}, SHmax={SHMAX}, Shmin={SHMIN}")


## Compute Kirsch borehole-wall stresses


In [ ]:
wall = NearWellboreStressesCalculation.calculate_kirsch_borehole_wall_stresses(
    shmin=SHMIN,
    shmax=SHMAX,
    svert=SVERT,
    pore_pressure=PORE_PRESSURE,
    shmax_azimuth=SHMAX_AZIMUTH,
    mud_pressure=MUD_PRESSURE,
    theta=theta,
    poisson_ratio_static=POISSON_RATIO_STATIC,
    borehole_deviation=BOREHOLE_DEVIATION,
    borehole_azimuth=BOREHOLE_AZIMUTH,
)

principals = NearWellboreStressesCalculation.calculate_principal_stresses_analytical(
    sigma_tt=wall.sigma_tt,
    sigma_zz=wall.sigma_zz,
    sigma_tz=wall.sigma_tz,
)

print("sigma_rr range (psi):", wall.sigma_rr.min(), "–", wall.sigma_rr.max())
print("sigma_tt range (psi):", wall.sigma_tt.min(), "–", wall.sigma_tt.max())
print("sigma_zz range (psi):", wall.sigma_zz.min(), "–", wall.sigma_zz.max())
print("sigma_1  range (psi):", principals.sigma_1.min(), "–", principals.sigma_1.max())


## Polar plot — stress components around the borehole


In [ ]:
theta_rad = np.deg2rad(theta)

fig, axes = plt.subplots(1, 3, figsize=(15, 5), subplot_kw=dict(projection="polar"))

# Tangential (hoop) stress — most critical for breakout / breakdown
axes[0].plot(theta_rad, wall.sigma_tt / 1000, "b-", lw=2)
axes[0].fill_between(theta_rad, 0, wall.sigma_tt / 1000, alpha=0.25, color="b")
axes[0].set_title(r"$\sigma_{\theta\\theta}$ (ksi)", pad=20)
axes[0].set_theta_zero_location("N")
axes[0].set_theta_direction(-1)  # clockwise

# Axial stress
axes[1].plot(theta_rad, wall.sigma_zz / 1000, "g-", lw=2)
axes[1].fill_between(theta_rad, 0, wall.sigma_zz / 1000, alpha=0.25, color="g")
axes[1].set_title(r"$\sigma_{zz}$ (ksi)", pad=20)
axes[1].set_theta_zero_location("N")
axes[1].set_theta_direction(-1)

# Maximum principal stress on the wall
axes[2].plot(theta_rad, principals.sigma_1 / 1000, "r-", lw=2)
axes[2].fill_between(theta_rad, 0, principals.sigma_1 / 1000, alpha=0.25, color="r")
axes[2].set_title(r"$\sigma_1$ principal (ksi)", pad=20)
axes[2].set_theta_zero_location("N")
axes[2].set_theta_direction(-1)

plt.suptitle(
    f"Kirsch wall stresses | SHmax az={SHMAX_AZIMUTH}° | "
    f"dev={BOREHOLE_DEVIATION}° az={BOREHOLE_AZIMUTH}° | "
    f"Pw={MUD_PRESSURE:.0f} psi",
    fontsize=11,
)
plt.tight_layout()
plt.show()


## Cartesian view — stress vs azimuth


In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(theta, wall.sigma_tt / 1000, "b-", label=r"$\sigma_{\\theta\\theta}$")
ax.plot(theta, wall.sigma_zz / 1000, "g-", label=r"$\sigma_{zz}$")
ax.plot(theta, wall.sigma_rr / 1000, "k--", label=r"$\sigma_{rr}$")
ax.plot(theta, principals.sigma_1 / 1000, "r-", lw=2, label=r"$\sigma_1$")
ax.plot(theta, principals.sigma_2 / 1000, "m-", lw=2, label=r"$\sigma_2$")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel(r"$\\theta$ (deg from Top-of-Hole)")
ax.set_ylabel("Stress (ksi)")
ax.set_title("Borehole-wall stresses vs azimuth")
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 360)
plt.tight_layout()
plt.show()


## Compare vertical vs horizontal well

Same far-field stresses; only borehole deviation changes.


In [ ]:
def run_kirsch(deviation):
    w = NearWellboreStressesCalculation.calculate_kirsch_borehole_wall_stresses(
        shmin=SHMIN, shmax=SHMAX, svert=SVERT,
        pore_pressure=PORE_PRESSURE, shmax_azimuth=SHMAX_AZIMUTH,
        mud_pressure=MUD_PRESSURE, theta=theta,
        poisson_ratio_static=POISSON_RATIO_STATIC,
        borehole_deviation=deviation, borehole_azimuth=BOREHOLE_AZIMUTH,
    )
    p = NearWellboreStressesCalculation.calculate_principal_stresses_analytical(
        sigma_tt=w.sigma_tt, sigma_zz=w.sigma_zz, sigma_tz=w.sigma_tz,
    )
    return w, p

wall_v, prin_v = run_kirsch(0.0)
wall_h, prin_h = run_kirsch(90.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5), subplot_kw=dict(projection="polar"))

axes[0].plot(theta_rad, prin_v.sigma_1 / 1000, "r-", lw=2, label=r"$\sigma_1$")
axes[0].plot(theta_rad, prin_v.sigma_2 / 1000, "b-", lw=2, label=r"$\sigma_2$")
axes[0].set_title("Vertical well (dev=0°)", pad=20)
axes[0].set_theta_zero_location("N")
axes[0].set_theta_direction(-1)
axes[0].legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))

axes[1].plot(theta_rad, prin_h.sigma_1 / 1000, "r-", lw=2, label=r"$\sigma_1$")
axes[1].plot(theta_rad, prin_h.sigma_2 / 1000, "b-", lw=2, label=r"$\sigma_2$")
axes[1].set_title("Horizontal well (dev=90°)", pad=20)
axes[1].set_theta_zero_location("N")
axes[1].set_theta_direction(-1)

plt.suptitle("Principal wall stresses — vertical vs horizontal", fontsize=12)
plt.tight_layout()
plt.show()


## Notes

- **$\\theta = 0°$** is Top-of-Hole (TOH).
- For a vertical well, breakouts form where $\\sigma_{\\theta\\theta}$ is maximum (typically along Shmin direction).
- Tensile fractures initiate where $\\sigma_{\\theta\\theta}$ is minimum (along SHmax).
- Change `BOREHOLE_DEVIATION`, `SHMAX_AZIMUTH` or mud pressure above and re-run to explore the stress state.
